# ⚽ Análisis y Predicción de Partidos de Fútbol

Este notebook sirve como guía interactiva para comprender y probar de forma individual el funcionamiento del **Modelo Predictivo de Fútbol** desarrollado en el proyecto.

A lo largo de este archivo, veremos cómo cargar los ratings ELO de cada selección calculados a partir de los datos históricos y cómo usarlos para estimar los goles esperados (xG) y calcular las probabilidades de partido (Local, Empate o Visitante) aplicando la Distribución de Poisson.

### 1. Configuración de Importaciones

En esta celda importamos la librería `pandas` para manipular tablas de datos, y los módulos locales del proyecto (`src`) que contienen las funciones de conexión, el cálculo de ratings ELO y la simulación matemática con Poisson.

In [1]:
import pandas as pd

from src.utils import obtener_conexion
from src.poisson import predecir_probabilidades_goles, obtener_probabilidades_1X2

### 2. Exploración de los Ratings ELO de las Selecciones

Para entender qué tan fuerte es cada selección nacional en un momento dado, utilizamos los ratings ELO. 

En la siguiente celda, abrimos una conexión a nuestra base de datos SQLite local, consultamos la tabla `ratings_elo` ordenándola de mayor a menor rating para visualizar el Top 10 de las selecciones actuales, y cerramos la conexión de forma segura.

In [2]:
conexion = obtener_conexion()
df_elo = pd.read_sql_query("""
    SELECT team AS Selección, rating AS [Rating ELO], confederation AS Confederación, fifa_rank AS [Ranking FIFA]
    FROM ratings_elo
    ORDER BY rating DESC
    LIMIT 10
""", conexion)
conexion.close()

df_elo

,Selección,Rating ELO,Confederación,Ranking FIFA
0,Argentina,2096.09,CONMEBOL,3
1,France,2069.97,UEFA,1
2,Spain,2069.20,UEFA,2
3,Mexico,2052.50,CONCACAF,15
4,England,2038.56,UEFA,4
5,Norway,2027.36,UEFA,44
6,Morocco,2026.93,CAF,8
7,Germany,1995.27,UEFA,10
8,USA,1975.34,CONCACAF,16
9,Colombia,1971.32,CONMEBOL,13


### 3. Simulación de un Enfrentamiento Directo

Aquí ponemos a prueba nuestro motor de predicción principal.

Definimos un partido entre dos equipos (por ejemplo, Argentina y Francia) en cancha neutral (peso de Mundial). La función `predecir_probabilidades_goles` calcula los goles esperados (xG o lambdas) para cada equipo a partir de la diferencia de sus ratings ELO y genera una matriz de probabilidad de resultados exactos. 

Luego, con `obtener_probabilidades_1X2`, sumamos las probabilidades de la matriz para obtener la probabilidad final de victoria local, empate y victoria visitante.

In [3]:
equipo_A = "Argentina"
equipo_B = "France"

# Calculamos las tasas esperadas de goles (lambdas) y la matriz de probabilidad de goles exactos
xg_A, xg_B, matriz = predecir_probabilidades_goles(equipo_A, equipo_B, es_neutral=True)

# Obtenemos las probabilidades de victoria A, empate y victoria B
prob_A, prob_empate, prob_B = obtener_probabilidades_1X2(matriz)

print(f"=== Predicción de Partido Neutral ===")
print(f"{equipo_A} vs {equipo_B}\n")
print(f"Goles Esperados (xG) para {equipo_A}: {xg_A:.2f}")
print(f"Goles Esperados (xG) para {equipo_B}: {xg_B:.2f}\n")
print(f"Probabilidad de Victoria de {equipo_A}: {prob_A*100:.1f}%")
print(f"Probabilidad de Empate: {prob_empate*100:.1f}%")
print(f"Probabilidad de Victoria de {equipo_B}: {prob_B*100:.1f}%")

=== Predicción de Partido Neutral ===
Argentina vs France

Goles Esperados (xG) para Argentina: 1.33
Goles Esperados (xG) para France: 0.96

Probabilidad de Victoria de Argentina: 45.3%
Probabilidad de Empate: 27.9%
Probabilidad de Victoria de France: 26.9%
